In [3]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec
import fasttext.util
import fasttext
import fasttext.util
import gzip
import os
import pandas as pd
import anthropic
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거
import requests
import json

In [4]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_excel('../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../data/info.csv')
api_key = api.loc[0][1]

In [4]:
df = df[['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']]

In [5]:
pi_list = df.PI.dropna().tolist()
len(pi_list)
pi_list = list(dict.fromkeys(pi_list))

In [6]:
pi_text = ' '.join(pi_list)

In [ ]:
len(pi_text)

In [7]:
import os
import json
import logging
from datetime import datetime
from typing import List, Dict, Optional
import asyncio
import pandas as pd
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from tqdm.asyncio import tqdm as tqdm_asyncio
import nest_asyncio
nest_asyncio.apply()  # Apply the patch for nested event loops




In [ ]:
from konlpy.tag import Okt
from collections import Counter
import re

def analyze_dental_text(text_list):
    okt = Okt()
    
    # 전처리: 리스트의 모든 텍스트를 하나의 문자열로 합치기
    all_text = ' '.join(text_list)
    
    # '*' 기호 제거 및 특수문자 처리
    all_text = all_text.replace('*', ' ')
    
    # 한글과 영어 분리하여 분석
    korean_pattern = re.compile('[가-힣]+')
    english_pattern = re.compile('[a-zA-Z]+')
    
    # 한글 단어 추출
    korean_words = []
    for word in korean_pattern.finditer(all_text):
        korean_words.append(word.group())
    
    # 영어 단어 추출
    english_words = []
    for word in english_pattern.finditer(all_text):
        english_words.append(word.group())
    
    # Okt로 한글 형태소 분석
    morphs = []
    for word in korean_words:
        morphs.extend(okt.pos(word))
    
    # 주요 단어 추출 (명사, 형용사)
    important_words = []
    for word, pos in morphs:
        if pos in ['Noun', 'Adjective']:
            important_words.append(word)
    
    # 단어 빈도수 계산
    korean_freq = Counter(important_words)
    english_freq = Counter(english_words)
    
    return {
        'korean_analysis': {
            'morphemes': morphs,
            'frequency': korean_freq.most_common(),
        },
        'english_analysis': {
            'words': english_words,
            'frequency': english_freq.most_common()
        }
    }

# 데이터 분석 실행
pi_list = df.PI.dropna().tolist()
pi_list = [i.replace('\n', ' ') for i in pi_list]
dental_records = pi_list

results = analyze_dental_text(dental_records)

# 결과 출력
print("=== 한글 분석 결과 ===")
print("\n형태소 분석:")
for morph, pos in results['korean_analysis']['morphemes'][:20]:  # 처음 20개만 출력
    print(f"{morph} ({pos})")

print("\n주요 한글 단어 빈도:")
for word, count in ['korean_analysis']['frequency']:
    print(f"{word}: {count}회")

print("\n=== 영어 분석 결과 ===")
for word, count in results['english_analysis']['frequency']:
    print(f"{word}: {count}회")

In [ ]:
import requests
import json
import time
import os
from math import ceil

def chunk_large_text(text, max_bytes=4000000, overlap_chars=1000):
    """
    대용량 텍스트를 청크로 나누어 처리합니다.
    기존보다 더 작은 청크 크기 사용 (8MB → 4MB)
    
    Args:
        text (str): 전체 텍스트
        max_bytes (int): 각 청크의 최대 바이트 크기
        overlap_chars (int): 청크 간 중복되는 문자 수
    
    Returns:
        list: 텍스트 청크 목록
    """
    chunks = []
    
    # 전체 텍스트를 바이트로 변환
    text_bytes = text.encode('utf-8')
    total_bytes = len(text_bytes)
    
    if total_bytes <= max_bytes:
        return [text]
    
    # 청크 수 계산 (50% 중첩 고려)
    num_chunks = ceil(total_bytes / (max_bytes * 0.5))
    
    # 문자 단위로 나누기 위한 계산
    chars_per_chunk = len(text) // num_chunks
    
    # 청크 분할
    start = 0
    while start < len(text):
        # 다음 청크의 끝 위치 계산
        end = min(start + chars_per_chunk + overlap_chars, len(text))
        
        # 문장 경계를 찾기
        if end < len(text):
            # 마침표, 느낌표, 물음표 뒤에 공백이 있는 위치 찾기
            sentence_end = max(
                text.rfind('. ', start, end),
                text.rfind('! ', start, end),
                text.rfind('? ', start, end),
                text.rfind('\n\n', start, end)  # 단락 구분자도 추가
            )
            
            # 문장 경계가 있으면 그 위치로 끝 조정
            if sentence_end > start + chars_per_chunk // 2:
                end = sentence_end + 2  # 문장 끝 + 공백 포함
        
        # 청크 추가
        chunk = text[start:end]
        chunks.append(chunk)
        
        # 다음 청크의 시작점 (중복 고려)
        start = end - overlap_chars if end < len(text) else end
    
    return chunks

def process_tmj_text_with_claude(api_key, text_chunks, base_prompt):
    """
    텍스트 청크를 Claude API를 사용하여 처리합니다.
    
    Args:
        api_key (str): Anthropic API 키
        text_chunks (list): 텍스트 청크 목록
        base_prompt (str): 프롬프트 베이스 템플릿
    
    Returns:
        list: 각 청크의 분석 결과
    """
    all_results = []
    
    for i, chunk in enumerate(text_chunks):
        print(f"청크 {i+1}/{len(text_chunks)} 처리 중... ({len(chunk.encode('utf-8'))} 바이트)")
        
        # 청크 크기 확인
        chunk_bytes = len(chunk.encode('utf-8'))
        if chunk_bytes > 4500000:  # 4.5MB 제한 (더 작게 설정)
            print(f"경고: 청크 {i+1}이 너무 큽니다 ({chunk_bytes} 바이트). 더 작게 분할합니다.")
            sub_chunks = chunk_large_text(chunk, 4000000, 1000)
            sub_results = process_tmj_text_with_claude(api_key, sub_chunks, base_prompt)
            all_results.extend(sub_results)
            continue
        
        # 프롬프트 준비
        full_prompt = base_prompt.replace("{texts}", chunk)
        
        # API 요청 설정
        headers = {
            "Content-Type": "application/json",
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01"
        }
        
        data = {
            "model": "claude-3-5-sonnet-20241022",
            "max_tokens": 4000,
            "temperature": 0,
            "system": "턱관절장애(TMJ) 진료 기록을 JSON 형식으로만 응답하세요. 설명이나 주석 없이 JSON 데이터만 반환하세요.",
            "messages": [
                {
                    "role": "user",
                    "content": full_prompt
                }
            ]
        }
        
        # API 요청
        try:
            response = requests.post(
                "https://api.anthropic.com/v1/messages",
                headers=headers,
                json=data
            )
            
            if response.status_code == 200:
                result = response.json()
                response_text = result['content'][0]['text']
                
                # 원시 응답 저장 (디버깅용)
                with open(f"tmj_raw_response_{i+1}.txt", "w", encoding="utf-8") as f:
                    f.write(response_text)
                
                # JSON 추출
                try:
                    # JSON 부분 추출 시도 (추출 방식 강화)
                    json_str = ""
                    
                    # 방법 1: ```json 코드 블록 찾기
                    if "```json" in response_text:
                        json_str = response_text.split("```json")[1].split("```")[0].strip()
                    
                    # 방법 2: 일반 ``` 코드 블록 찾기
                    elif "```" in response_text:
                        code_blocks = response_text.split("```")
                        # 첫 번째 코드 블록 내용 사용
                        if len(code_blocks) >= 3:  # 최소한 시작과 끝 태그가 있어야 함
                            json_str = code_blocks[1].strip()
                    
                    # 방법 3: 배열 형태 찾기 - 가장 바깥쪽 괄호 쌍 찾기
                    elif '[' in response_text and ']' in response_text:
                        start_idx = response_text.find('[')
                        end_idx = response_text.rfind(']') + 1
                        if start_idx != -1 and end_idx > start_idx:
                            json_str = response_text[start_idx:end_idx]
                    
                    # 방법 4: 객체 형태 찾기
                    elif '{' in response_text and '}' in response_text:
                        start_idx = response_text.find('{')
                        end_idx = response_text.rfind('}') + 1
                        if start_idx != -1 and end_idx > start_idx:
                            json_str = response_text[start_idx:end_idx]
                    
                    # JSON 문자열이 추출되었는지 확인
                    if json_str:
                        # 추출된 JSON 문자열 저장 (디버깅용)
                        with open(f"tmj_json_str_{i+1}.txt", "w", encoding="utf-8") as f:
                            f.write(json_str)
                        
                        # JSON 파싱
                        json_result = json.loads(json_str)
                        
                        # 결과가 리스트인지 확인하고 적절히 처리
                        if isinstance(json_result, list):
                            all_results.extend(json_result)
                            print(f"청크 {i+1}: {len(json_result)}개 항목 추출 성공")
                        else:
                            all_results.append(json_result)
                            print(f"청크 {i+1}: 1개 항목 추출 성공")
                        
                        # 결과 저장 (청크별)
                        with open(f"tmj_results_chunk_{i+1}.json", "w", encoding="utf-8") as f:
                            json.dump(json_result, f, ensure_ascii=False, indent=2)
                    else:
                        print(f"청크 {i+1}에서 JSON을 찾을 수 없습니다.")
                        # 2차 시도: 더 작은 청크로 분할
                        if len(chunk) > 2000:
                            print("더 작은 청크로 분할하여 재시도합니다.")
                            mid = len(chunk) // 2
                            # 문장 경계 찾기
                            boundary = max(
                                chunk.rfind('. ', 0, mid),
                                chunk.rfind('! ', 0, mid),
                                chunk.rfind('? ', 0, mid),
                                chunk.rfind('\n\n', 0, mid)
                            )
                            
                            if boundary > 0:
                                mid = boundary + 2
                            
                            sub_chunks = [chunk[:mid], chunk[mid:]]
                            sub_results = process_tmj_text_with_claude(api_key, sub_chunks, base_prompt)
                            all_results.extend(sub_results)
                
                except (json.JSONDecodeError, IndexError) as e:
                    print(f"청크 {i+1} JSON 파싱 오류: {str(e)}")
                    print("오류가 발생한 JSON 문자열의 일부:")
                    if json_str:
                        print(json_str[:100] + "..." if len(json_str) > 100 else json_str)
                    
                    # 기본 중첩 구조 수정 시도
                    if json_str:
                        try:
                            # 중첩된 따옴표 이스케이프 문제 수정 시도
                            fixed_json = json_str.replace('\\"', '"').replace('\\\\', '\\')
                            json_result = json.loads(fixed_json)
                            
                            if isinstance(json_result, list):
                                all_results.extend(json_result)
                                print(f"청크 {i+1}: 수정 후 {len(json_result)}개 항목 추출 성공")
                            else:
                                all_results.append(json_result)
                                print(f"청크 {i+1}: 수정 후 1개 항목 추출 성공")
                            
                            with open(f"tmj_results_chunk_{i+1}_fixed.json", "w", encoding="utf-8") as f:
                                json.dump(json_result, f, ensure_ascii=False, indent=2)
                        except json.JSONDecodeError:
                            print("JSON 수정 시도 실패")
            else:
                print(f"청크 {i+1} 에러 코드: {response.status_code}")
                print(f"에러 메시지: {response.text}")
        
        except Exception as e:
            print(f"청크 {i+1} 예외 발생: {str(e)}")
        
        # API 요청 간 딜레이
        if i < len(text_chunks) - 1:
            time.sleep(2)
    
    return all_results

# 원본 프롬프트 템플릿 유지 (요청에 따라 프롬프트를 줄이지 않음)
tmj_prompt_template = """
다음은 턱관절장애(TMJ) 및 저작근 장애에 대한 진료 기록(PI 텍스트)입니다. 
문서에는 K07.65(퇴행성 관절염), K07.66(저작근의 장애), K07.63(턱관절 통증) 등의 진단 코드, 
파노라마/CT 촬영, 측두하악장애분석검사, 물리치료, 약물 처방, 경과 관찰 등의 내용이 포함될 수 있습니다.

아래 텍스트를 분석하여, 다음 14개 필드를 JSON 형식으로 추출해주세요:

1. onset (발현 시기)  
- 예: "3개월 전", "2023년 1월", "발병 시기 미상"  
- 텍스트에서 구체적으로 언급된 경우만 추출하고, 없으면 `""`(빈 문자열)

2. pattern (증상 양상)  
- "constant" (지속성), "intermittent" (간헐성), "progressive" (점진적 악화)  
- 명확히 언급된 경우에만 지정. 없으면 `""`

3. aggravating_factors (악화 요인 목록, 배열)  
- 예: ["딱딱한 음식", "스트레스", "이 악물기"]  
- 여러 개라면 배열에 순서대로 담고, 없으면 `[]`

4. status (현재 상태)  
- "improving" (호전), "unchanged" (변화 없음), "worsening" (악화)  
- 명시되지 않으면 `""`

5. TMJ_PI_desc (진단/검사 항목, 배열)  
- 파노라마, CT, 측두하악장애분석검사, 초음파, T-scan 등 실시된 검사 이름을 배열로 적습니다.  
- 예: ["파노라마", "Cone Beam CT"]

6. TMJ_PI_treatment (물리치료 항목, 배열)  
- 분사신장치료, 전기자극치료, 복합자극치료, 물리치료 등  
- 예: ["측두하악관절자극요법-단순", "분사신장치료"]

7. drug_treatment (약물치료 항목, 배열)  
- 예: ["소론도정(프레드니솔론)", "페리슨정(에페리손염산염)"]  
- 복용 방법, 용량, 횟수 등은 이 필드가 아닌 `medication_prescription`에 기입

8. closing_dentalgear_desc (교합치료 항목, 배열)  
- 교합안정장치(Splint), 교합조정, 보톡스(교합개선 목적) 등 교합 관련 치료가 있으면 적습니다.  
- 없으면 `[]`

9. PI_check (경과관찰 항목, 배열)  
- "증상 체크 [2주후]", "1개월 후 재내원", "재평가 예정" 등 주기적 검진/재평가 계획  
- 예: ["증상 ck [2주후]"]

10. PI_diagnosis_jojint (턱관절 진단 내용, 문자열)  
- K07.65 퇴행성 관절염, K07.63 턱관절 통증, K07.66 저작근 장애 등 진단명  
- 예: "퇴행성 관절염 (K07.65)"

11. physical_therapy (저작근 장애(K07.66) 물리치료 등, 문자열)  
- 저작근 장애를 치료하기 위해 시행된 물리치료나 자극요법(단순/전기/복합), 분사신장치료 등의 종합 요약  
- 예: "측두하악관절 단순/전기/복합 자극, 분사신장치료"

12. occlusal_treatment (교합안정장치(Splint) 등 장치치료, 문자열)  
- 예: "APS, SS 장치 장착 및 조정"  
- 교합 조정, 장치 제작 등

13. medication_prescription (약물 처방 상세, 문자열)  
- "페리슨정(에페리손염산염) 1/1회/14일 :: 취침 직전 복용" 처럼, 복용 용법/횟수/기간 등을 구체적으로 적습니다.  
- 여러 개라면 문장으로 나열  
- 예: "페리슨정(에페리손염산염) - 1/1회/14일, 소론도정(프레드니솔론) - 1/1회/14일"

14. other_treatment (그 외 보톡스 시술, 악관절 강세척술, 교합조정, 발치 등, 문자열)  
- "보톡스 시술" 등  
- 텍스트에서 확인되면 구체적으로 작성, 없으면 `""`

응답은 다음과 같이 JSON 배열 형태로 주세요:
```json
[
  {
    "onset": "...",
    "pattern": "...",
    "aggravating_factors": [...],
    "status": "...",
    "TMJ_PI_desc": [...],
    "TMJ_PI_treatment": [...],
    "drug_treatment": [...],
    "closing_dentalgear_desc": [...],
    "PI_check": [...],
    "PI_diagnosis_jojint": "...",
    "physical_therapy": "...",
    "occlusal_treatment": "...",
    "medication_prescription": "...",
    "other_treatment": "..."
  }
]
```

설명이나 주석 없이 JSON만 반환해주세요.

분석할 텍스트:
{texts}
"""

# 분석 결과 통합
def combine_tmj_results(results):
    """
    여러 청크의 TMJ 분석 결과를 하나로 통합합니다.
    
    Args:
        results (list): 각 청크의 분석 결과 목록
    
    Returns:
        list: 통합된 분석 결과
    """
    # 단순히 모든 결과를 합치기
    return results

# 누락된 청크 결과 통합
def combine_chunk_files():
    """
    개별적으로 저장된 청크 결과 파일들을 통합합니다.
    """
    all_results = []
    
    # tmj_results_chunk_*.json 파일 찾기
    chunk_files = [f for f in os.listdir('.') if f.startswith('tmj_results_chunk_') and f.endswith('.json')]
    chunk_files.sort(key=lambda x: int(x.split('_')[-1].split('.')[0]))
    
    if not chunk_files:
        print("청크 결과 파일을 찾을 수 없습니다.")
        return all_results
    
    print(f"{len(chunk_files)}개의 청크 결과 파일을 통합합니다.")
    
    for file_name in chunk_files:
        try:
            with open(file_name, 'r', encoding='utf-8') as f:
                chunk_result = json.load(f)
            
            if isinstance(chunk_result, list):
                all_results.extend(chunk_result)
                print(f"{file_name}: {len(chunk_result)}개 항목 추가")
            else:
                all_results.append(chunk_result)
                print(f"{file_name}: 1개 항목 추가")
        
        except Exception as e:
            print(f"{file_name} 처리 중 오류 발생: {str(e)}")
    
    # 결과 저장
    with open('tmj_analysis_combined.json', 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    
    print(f"통합 완료. 총 {len(all_results)}개 항목이 tmj_analysis_combined.json에 저장되었습니다.")
    
    return all_results

# 메인 프로그램
def process_large_tmj_text(api_key, pi_text, output_file):
    """
    대용량 TMJ 텍스트를 처리합니다.
    
    Args:
        api_key (str): Anthropic API 키
        output_file (str): 출력 JSON 파일 경로
    """
    # pi_text 변수 사용 (글로벌 변수로 가정)
    # 텍스트를 청크로 분할 (더 작은 크기로)
    chunks = chunk_large_text(pi_text, max_bytes=4000000, overlap_chars=1000)
    print(f"텍스트가 {len(chunks)}개 청크로 분할되었습니다.")
    
    # 각 청크 크기 출력
    for i, chunk in enumerate(chunks):
        chunk_bytes = len(chunk.encode('utf-8'))
        print(f"청크 {i+1}: {chunk_bytes:,} 바이트 ({chunk_bytes/1024/1024:.2f} MB)")
    
    # Claude API로 텍스트 처리 (원본 프롬프트 사용)
    print("\n텍스트 분석 시작...")
    results = process_tmj_text_with_claude(api_key, chunks, tmj_prompt_template)
    
    # 결과 통합
    combined_results = combine_tmj_results(results)
    
    # 결과 저장
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(combined_results, f, ensure_ascii=False, indent=2)
    
    print(f"\n분석 완료! 결과가 {output_file}에 저장되었습니다.")
    print(f"총 {len(combined_results)}개의 TMJ 진료 기록이 분석되었습니다.")
    
    # 결과가 없는 경우 파일 통합 시도
    if len(combined_results) == 0:
        print("\n자동으로 청크 파일 통합을 시도합니다...")
        combine_chunk_files()

# 사용 예시
if __name__ == "__main__":
    api_key = api_key
    
    # 텍스트 파일에서 읽기
    
    output_file = "tmj_analysis.json"
    
    # try:
    #     with open(input_file, 'r', encoding='utf-8') as f:
    #         input_text = f.read()
        
    process_large_tmj_text(api_key, pi_text, output_file)
    
        # 이미 생성된 파일 통합 시도
    print("\n파일 통합을 시도합니다...")
    combine_chunk_files()

In [ ]:
import requests
import json
import time
import os
from math import ceil

# PI 폴더 생성 (파일 저장용)
PI_FOLDER = "PI"
os.makedirs(PI_FOLDER, exist_ok=True)

def chunk_large_text(text, max_bytes=300000, overlap_chars=1000):
    """
    대용량 텍스트를 청크로 나누어 처리합니다.
    기본 청크 크기를 300KB로 설정하고, 청크 간 1000자 중복 처리합니다.
    
    Args:
        text (str): 전체 텍스트
        max_bytes (int): 각 청크의 최대 바이트 크기 (기본 300,000 바이트)
        overlap_chars (int): 청크 간 중복되는 문자 수
    
    Returns:
        list: 텍스트 청크 목록
    """
    chunks = []
    text_bytes = text.encode('utf-8')
    total_bytes = len(text_bytes)
    
    if total_bytes <= max_bytes:
        return [text]
    
    num_chunks = ceil(total_bytes / (max_bytes * 0.5))
    chars_per_chunk = len(text) // num_chunks
    start = 0
    while start < len(text):
        end = min(start + chars_per_chunk + overlap_chars, len(text))
        if end < len(text):
            sentence_end = max(
                text.rfind('. ', start, end),
                text.rfind('! ', start, end),
                text.rfind('? ', start, end),
                text.rfind('\n\n', start, end)
            )
            if sentence_end > start + chars_per_chunk // 2:
                end = sentence_end + 2
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap_chars if end < len(text) else end
    return chunks

def process_tmj_text_with_claude(api_key, text_chunks, base_prompt):
    """
    텍스트 청크를 Claude API를 사용하여 처리하고, 응답 파일들을 PI 폴더에 저장합니다.
    
    Args:
        api_key (str): Anthropic API 키
        text_chunks (list): 텍스트 청크 목록
        base_prompt (str): 프롬프트 베이스 템플릿
    
    Returns:
        list: 각 청크의 분석 결과
    """
    all_results = []
    
    for i, chunk in enumerate(text_chunks):
        print(f"청크 {i+1}/{len(text_chunks)} 처리 중... ({len(chunk.encode('utf-8')):,} 바이트)")
        full_prompt = base_prompt.replace("{texts}", chunk)
        
        headers = {
            "Content-Type": "application/json",
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01"
        }
        
        data = {
            "model": "claude-3-5-sonnet-20241022",
            "max_tokens": 4000,
            "temperature": 0,
            "system": "턱관절장애(TMJ) 진료 기록을 JSON 형식으로만 응답하세요. 설명이나 주석 없이 JSON 데이터만 반환하세요.",
            "messages": [
                {
                    "role": "user",
                    "content": full_prompt
                }
            ]
        }
        
        max_retries = 3
        retry_delay = 10  # 초기 대기 시간 (초)
        
        for retry in range(max_retries):
            try:
                print(f"API 요청 시도 {retry + 1}/{max_retries}...")
                response = requests.post(
                    "https://api.anthropic.com/v1/messages",
                    headers=headers,
                    json=data
                )
                
                if response.status_code == 200:
                    result = response.json()
                    response_text = result['content'][0]['text']
                    
                    # PI 폴더에 원시 응답 저장
                    with open(os.path.join(PI_FOLDER, f"tmj_raw_response_{i+1}.txt"), "w", encoding="utf-8") as f:
                        f.write(response_text)
                    
                    json_str = ""
                    if "```json" in response_text:
                        json_str = response_text.split("```json")[1].split("```")[0].strip()
                    elif "```" in response_text:
                        code_blocks = response_text.split("```")
                        if len(code_blocks) >= 3:
                            json_str = code_blocks[1].strip()
                    elif '[' in response_text and ']' in response_text:
                        start_idx = response_text.find('[')
                        end_idx = response_text.rfind(']') + 1
                        if start_idx != -1 and end_idx > start_idx:
                            json_str = response_text[start_idx:end_idx]
                    elif '{' in response_text and '}' in response_text:
                        start_idx = response_text.find('{')
                        end_idx = response_text.rfind('}') + 1
                        if start_idx != -1 and end_idx > start_idx:
                            json_str = response_text[start_idx:end_idx]
                    
                    if json_str:
                        with open(os.path.join(PI_FOLDER, f"tmj_json_str_{i+1}.txt"), "w", encoding="utf-8") as f:
                            f.write(json_str)
                        json_result = json.loads(json_str)
                        if isinstance(json_result, list):
                            all_results.extend(json_result)
                            print(f"청크 {i+1}: {len(json_result)}개 항목 추출 성공")
                        else:
                            all_results.append(json_result)
                            print(f"청크 {i+1}: 1개 항목 추출 성공")
                        with open(os.path.join(PI_FOLDER, f"tmj_results_chunk_{i+1}.json"), "w", encoding="utf-8") as f:
                            json.dump(json_result, f, ensure_ascii=False, indent=2)
                        break
                    else:
                        print(f"청크 {i+1}에서 JSON을 찾을 수 없습니다.")
                        if retry < max_retries - 1:
                            print(f"{retry_delay}초 후 재시도합니다...")
                            time.sleep(retry_delay)
                            continue
                        else:
                            print("최대 재시도 횟수 초과. 다음 청크로 넘어갑니다.")
                elif response.status_code in [429, 500, 502, 503, 504, 520]:
                    print(f"청크 {i+1} 에러 코드: {response.status_code}")
                    print(f"서버 오류 또는 속도 제한. {retry_delay}초 후 재시도합니다...")
                    time.sleep(retry_delay)
                    retry_delay *= 2  # 지수 백오프 적용
                    continue
                else:
                    print(f"청크 {i+1} 에러 코드: {response.status_code}")
                    print(f"에러 메시지: {response.text}")
                    if retry < max_retries - 1:
                        print(f"{retry_delay}초 후 재시도합니다...")
                        time.sleep(retry_delay)
                        continue
                    else:
                        print("최대 재시도 횟수 초과. 다음 청크로 넘어갑니다.")
            except Exception as e:
                print(f"청크 {i+1} 예외 발생: {str(e)}")
                if retry < max_retries - 1:
                    print(f"{retry_delay}초 후 재시도합니다...")
                    time.sleep(retry_delay)
                    continue
                else:
                    print("최대 재시도 횟수 초과. 다음 청크로 넘어갑니다.")
        if i < len(text_chunks) - 1:
            sleep_time = 10
            print(f"다음 청크 처리를 위해 {sleep_time}초 대기 중...")
            time.sleep(sleep_time)
    return all_results

# 원본 프롬프트 템플릿 유지
tmj_prompt_template = """
다음은 턱관절장애(TMJ) 및 저작근 장애에 대한 진료 기록(PI 텍스트)입니다. 
문서에는 K07.65(퇴행성 관절염), K07.66(저작근의 장애), K07.63(턱관절 통증) 등의 진단 코드, 
파노라마/CT 촬영, 측두하악장애분석검사, 물리치료, 약물 처방, 경과 관찰 등의 내용이 포함될 수 있습니다.

아래 텍스트를 분석하여, 다음 14개 필드를 JSON 형식으로 추출해주세요:

1. onset (발현 시기)  
- 예: "3개월 전", "2023년 1월", "발병 시기 미상"  
- 텍스트에서 구체적으로 언급된 경우만 추출하고, 없으면 `""`(빈 문자열)

2. pattern (증상 양상)  
- "constant" (지속성), "intermittent" (간헐성), "progressive" (점진적 악화)  
- 명확히 언급된 경우에만 지정. 없으면 `""`

3. aggravating_factors (악화 요인 목록, 배열)  
- 예: ["딱딱한 음식", "스트레스", "이 악물기"]  
- 여러 개라면 배열에 순서대로 담고, 없으면 `[]`

4. status (현재 상태)  
- "improving" (호전), "unchanged" (변화 없음), "worsening" (악화)  
- 명시되지 않으면 `""`

5. TMJ_PI_desc (진단/검사 항목, 배열)  
- 파노라마, CT, 측두하악장애분석검사, 초음파, T-scan 등 실시된 검사 이름을 배열로 적습니다.  
- 예: ["파노라마", "Cone Beam CT"]

6. TMJ_PI_treatment (물리치료 항목, 배열)  
- 분사신장치료, 전기자극치료, 복합자극치료, 물리치료 등  
- 예: ["측두하악관절자극요법-단순", "분사신장치료"]

7. drug_treatment (약물치료 항목, 배열)  
- 예: ["소론도정(프레드니솔론)", "페리슨정(에페리손염산염)"]  
- 복용 방법, 용량, 횟수 등은 이 필드가 아닌 `medication_prescription`에 기입

8. closing_dentalgear_desc (교합치료 항목, 배열)  
- 교합안정장치(Splint), 교합조정, 보톡스(교합개선 목적) 등 교합 관련 치료가 있으면 적습니다.  
- 없으면 `[]`

9. PI_check (경과관찰 항목, 배열)  
- "증상 체크 [2주후]", "1개월 후 재내원", "재평가 예정" 등 주기적 검진/재평가 계획  
- 예: ["증상 ck [2주후]"]

10. PI_diagnosis_jojint (턱관절 진단 내용, 문자열)  
- K07.65 퇴행성 관절염, K07.63 턱관절 통증, K07.66 저작근 장애 등 진단명  
- 예: "퇴행성 관절염 (K07.65)"

11. physical_therapy (저작근 장애(K07.66) 물리치료 등, 문자열)  
- 저작근 장애를 치료하기 위해 시행된 물리치료나 자극요법(단순/전기/복합), 분사신장치료 등의 종합 요약  
- 예: "측두하악관절 단순/전기/복합 자극, 분사신장치료"

12. occlusal_treatment (교합안정장치(Splint) 등 장치치료, 문자열)  
- 예: "APS, SS 장치 장착 및 조정"  
- 교합 조정, 장치 제작 등

13. medication_prescription (약물 처방 상세, 문자열)  
- "페리슨정(에페리손염산염) 1/1회/14일 :: 취침 직전 복용" 처럼, 복용 용법/횟수/기간 등을 구체적으로 적습니다.  
- 여러 개라면 문장으로 나열  
- 예: "페리슨정(에페리손염산염) - 1/1회/14일, 소론도정(프레드니솔론) - 1/1회/14일"

14. other_treatment (그 외 보톡스 시술, 악관절 강세척술, 교합조정, 발치 등, 문자열)  
- "보톡스 시술" 등  
- 텍스트에서 확인되면 구체적으로 작성, 없으면 `""`

응답은 다음과 같이 JSON 배열 형태로 주세요:
```json
[
  {
    "onset": "...",
    "pattern": "...",
    "aggravating_factors": [...],
    "status": "...",
    "TMJ_PI_desc": [...],
    "TMJ_PI_treatment": [...],
    "drug_treatment": [...],
    "closing_dentalgear_desc": [...],
    "PI_check": [...],
    "PI_diagnosis_jojint": "...",
    "physical_therapy": "...",
    "occlusal_treatment": "...",
    "medication_prescription": "...",
    "other_treatment": "..."
  }
]
```

설명이나 주석 없이 JSON만 반환해주세요.

분석할 텍스트:
{texts}
"""

# 분석 결과 통합
def combine_tmj_results(results):
    """
    여러 청크의 TMJ 분석 결과를 하나로 통합합니다.
    
    Args:
        results (list): 각 청크의 분석 결과 목록
    
    Returns:
        list: 통합된 분석 결과
    """
    # 단순히 모든 결과를 합치기
    return results

# 누락된 청크 결과 통합
def combine_chunk_files():
    """
    PI 폴더에 저장된 청크 결과 파일들을 통합합니다.
    """
    all_results = []
    
    # PI 폴더 내의 tmj_results_chunk_*.json 파일 찾기
    chunk_files = [f for f in os.listdir(PI_FOLDER) if f.startswith('tmj_results_chunk_') and f.endswith('.json')]
    chunk_files.sort(key=lambda x: int(x.split('_')[-1].split('.')[0]))
    
    if not chunk_files:
        print("청크 결과 파일을 찾을 수 없습니다.")
        return all_results
    
    print(f"{len(chunk_files)}개의 청크 결과 파일을 통합합니다.")
    
    for file_name in chunk_files:
        try:
            file_path = os.path.join(PI_FOLDER, file_name)
            with open(file_path, 'r', encoding='utf-8') as f:
                chunk_result = json.load(f)
            
            if isinstance(chunk_result, list):
                all_results.extend(chunk_result)
                print(f"{file_name}: {len(chunk_result)}개 항목 추가")
            else:
                all_results.append(chunk_result)
                print(f"{file_name}: 1개 항목 추가")
        
        except Exception as e:
            print(f"{file_name} 처리 중 오류 발생: {str(e)}")
    
    # 통합 결과를 PI 폴더 내에 저장
    combined_file = os.path.join(PI_FOLDER, 'tmj_analysis_combined.json')
    with open(combined_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    
    print(f"통합 완료. 총 {len(all_results)}개 항목이 {combined_file}에 저장되었습니다.")
    
    return all_results


# 메인 프로그램
def process_large_tmj_text(api_key, pi_text, output_file):
    """
    대용량 TMJ 텍스트를 처리합니다.
    
    Args:
        api_key (str): Anthropic API 키
        pi_text (str): 분석할 텍스트
        output_file (str): 출력 JSON 파일 경로
    """
    # 텍스트를 청크로 분할 (훨씬 더 작은 크기로 500KB)
    chunks = chunk_large_text(pi_text, max_bytes=500000, overlap_chars=1000)
    print(f"텍스트가 {len(chunks)}개 청크로 분할되었습니다.")
    
    # 각 청크 크기 출력
    for i, chunk in enumerate(chunks):
        chunk_bytes = len(chunk.encode('utf-8'))
        print(f"청크 {i+1}: {chunk_bytes:,} 바이트 ({chunk_bytes/1024/1024:.2f} MB)")
    
    # Claude API로 텍스트 처리 (원본 프롬프트 사용)
    print("\n텍스트 분석 시작...")
    results = process_tmj_text_with_claude(api_key, chunks, tmj_prompt_template)
    
    # 결과 통합
    combined_results = combine_tmj_results(results)
    
    # 결과 저장
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(combined_results, f, ensure_ascii=False, indent=2)
    
    print(f"\n분석 완료! 결과가 {output_file}에 저장되었습니다.")
    print(f"총 {len(combined_results)}개의 TMJ 진료 기록이 분석되었습니다.")
    
    # 결과가 없는 경우 파일 통합 시도
    if len(combined_results) == 0:
        print("\n자동으로 청크 파일 통합을 시도합니다...")
        combine_chunk_files()

# 사용 예시
if __name__ == "__main__":
    api_key = api_key  # 실제 API 키로 교체
    output_file = "tmj_analysis.json"
    
    # 이미 pi_text 변수가 정의되어 있음을 가정
    process_large_tmj_text(api_key, pi_text, output_file)
    
    # 이미 생성된 파일 통합 시도
    print("\n파일 통합을 시도합니다...")
    combine_chunk_files()

In [ ]:
combine_chunk_files()

In [5]:
pi_words = pd.read_json('./PI/tmj_analysis_combined.json')

In [6]:

pi_words.TMJ_PI_treatment.tolist()

[['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료'],
 ['측두하악관절자극요법-단순자극', '측두하악관절자극요법-전기자극', '측두하악관절자극요법-복합자극', '분사신장치료'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료', '악관절고착해소술'],
 ['측두하악관절자극요법-단순자극',
  '측두하악관절자극요법-전기자극',
  '측두하악관절자극요법-복합자극',
  '분사신장치료',
  '악관절고착해소술'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료', '악관절고착해소술'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료', '악관절고착해소술'],
 ['측두하악관절자극요법-단순자극', '측두하악관절자극요법-전기자극', '측두하악관절자극요법-복합자극', '분사신장치료'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료', '악관절 고착 해소술'],
 ['측두하악관절자극요법-단순', '측두하악관절자극요법-전기', '측두하악관절자극요법-복합', '분사신장치료', '물리치료'],
 ['측두하악관

In [8]:
import collections

# 각 열별로 고유 요소와 등장 빈도를 딕셔너리 형태로 저장하는 객체
frequency_by_column = {}

for col in pi_words.columns:
    counter = collections.Counter()
    for cell in pi_words[col]:
        # cell이 리스트면 각 원소를, 아니면 단일 값을 리스트로 취급
        elements = cell if isinstance(cell, list) else [cell]
        for elem in elements:
            # 문자열이 아닐 경우 건너뜁니다.
            if not isinstance(elem, str):
                continue
            # 먼저 콤마(,) 기준으로 분리하고 좌우 공백 제거
            tokens = [token.strip() for token in elem.split(',')]
            for token in tokens:
                # 만약 토큰 내에 슬래시(/)가 있다면 추가 분리하여 각각 카운트
                if '/' in token:
                    sub_tokens = [sub.strip() for sub in token.split('/')]
                    for sub in sub_tokens:
                        if sub:
                            counter[sub] += 1
                else:
                    if token:
                        counter[token] += 1
    frequency_by_column[col] = dict(counter)

print(frequency_by_column)


{'onset': {}, 'pattern': {'intermittent': 2, 'progressive': 1}, 'aggravating_factors': {'딱딱한 음식': 23, '스트레스': 15, '이 악물기': 20, '찬 음식': 1, '단 음식': 1}, 'status': {'improving': 4}, 'TMJ_PI_desc': {'파노라마': 32, 'Cone Beam CT': 32, '측두하악장애분석검사': 32, '초음파': 32, 'T-scan': 29, '파노라마(특수)': 10, '동기능적교합검사': 7, 'PCR': 1}, 'TMJ_PI_treatment': {'측두하악관절자극요법-단순': 29, '측두하악관절자극요법-전기': 29, '측두하악관절자극요법-복합': 29, '분사신장치료': 32, '측두하악관절자극요법-단순자극': 3, '측두하악관절자극요법-전기자극': 3, '측두하악관절자극요법-복합자극': 3, '악관절고착해소술': 8, '악관절 고착 해소술': 4, '물리치료': 2, '악관절강세척술': 2}, 'drug_treatment': {'페리슨정(에페리손염산염)': 30, '리보트릴정(클로나제팜)': 30, '소론도정(프레드니솔론)': 30, '세크로정(아세클로페낙)': 29, '휴모리드정(모사프리드시트르산염수화물)': 1, '휴모리드정5mg(모사프리드시트르산염수화물)': 16, '디푸루칸건조시럽(플루코나졸)': 2, '휴모리드정': 2, '뉴론틴캡슐300밀리그램(가바펜틴)': 5, '아목사정625밀리그램(아목시실린수화물-클라불란산칼륨)': 1, '뉴론틴정600밀리그램(가바펜틴)': 1, '살라겐정(필로카르핀염산염)': 1, '알마겔정(알마게이트)': 1, '휴온스아목시크라정625밀리그램': 1, '테그레톨정200밀리그램': 1, '아목클정625밀리그램(아목시실린·클라불란산칼륨)': 1, '아목클정625밀리그램': 1, '알마겔정': 3, '뉴론틴캡슐300밀리그램': 1, '리보트릴정': 2, '소론도정': 2, '페리슨정

In [9]:
frequency_by_column.keys()


dict_keys(['onset', 'pattern', 'aggravating_factors', 'status', 'TMJ_PI_desc', 'TMJ_PI_treatment', 'drug_treatment', 'closing_dentalgear_desc', 'PI_check', 'PI_diagnosis_jojint', 'physical_therapy', 'occlusal_treatment', 'medication_prescription', 'other_treatment'])

In [10]:
# 또는 방법 2: from_dict 사용
pd.DataFrame.from_dict(frequency_by_column['aggravating_factors'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
딱딱한 음식,23
이 악물기,20
스트레스,15
찬 음식,1
단 음식,1


In [11]:
# 또는 방법 2: from_dict 사용
pd.DataFrame.from_dict(frequency_by_column['TMJ_PI_desc'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
파노라마,32
Cone Beam CT,32
측두하악장애분석검사,32
초음파,32
T-scan,29
파노라마(특수),10
동기능적교합검사,7
PCR,1


In [12]:
# 또는 방법 2: from_dict 사용
pd.DataFrame.from_dict(frequency_by_column['TMJ_PI_treatment'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
분사신장치료,32
측두하악관절자극요법-단순,29
측두하악관절자극요법-전기,29
측두하악관절자극요법-복합,29
악관절고착해소술,8
악관절 고착 해소술,4
측두하악관절자극요법-단순자극,3
측두하악관절자극요법-전기자극,3
측두하악관절자극요법-복합자극,3
물리치료,2


In [13]:
pd.DataFrame.from_dict(frequency_by_column['drug_treatment'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
페리슨정(에페리손염산염),30
소론도정(프레드니솔론),30
리보트릴정(클로나제팜),30
세크로정(아세클로페낙),29
휴모리드정5mg(모사프리드시트르산염수화물),16
뉴론틴캡슐300밀리그램(가바펜틴),5
알마겔정,3
휴모리드정,2
디푸루칸건조시럽(플루코나졸),2
리보트릴정,2


In [14]:
pd.DataFrame.from_dict(frequency_by_column['closing_dentalgear_desc'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
APS Splint,30
SS Splint,30
이갈이장치,30
코골이장치,4
구강보호장치,3
트랙션장치,3
교합안정장치,2
이갈이 장치,2
APS,2
SS,2


In [15]:
pd.DataFrame.from_dict(frequency_by_column['PI_check'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
증상 ck [2주후],21
장치 ck [1개월후],16
증상 ck [1주후],11
장치 ck [2주후],7
근육두께ck [1개월후],6
근육두께ck [6주후],4
증상 체크 [2주후],4
근육두께ck [3주후],3
물리치료 [1개월후],3
증상 ck [1개월후],3


In [16]:
pd.DataFrame.from_dict(frequency_by_column['PI_diagnosis_jojint'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
턱관절의 퇴행성 관절염 (K07.65),32


In [17]:
pd.DataFrame.from_dict(frequency_by_column['physical_therapy'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
측두하악관절 단순,32
전기,32
분사신장치료,32
복합 자극,19
복합 자극요법,13
악관절고착해소술,5
악관절 고착 해소술,4
물리치료,4
초음파 치료,1


In [18]:
pd.DataFrame.from_dict(frequency_by_column['occlusal_treatment'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
APS,20
SS 장치 장착 및 조정,19
SS Splint,11
APS Splint,7
이갈이장치 장착 및 조정,6
APS Splint 장착 및 조정,5
SS,1
SS Splint 장착 및 조정,1
이갈이장치,1
코골이장치 장착 및 조정,1


In [19]:
pd.DataFrame.from_dict(frequency_by_column['medication_prescription'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
1회,72
페리슨정(에페리손염산염) - 1,27
14일 :: 취침 직전 복용,27
소론도정(프레드니솔론) - 1,16
2회,16
저녁 식후 30분 복용,13
7일 :: 취침 직전 복용,11
세크로정(아세클로페낙) - 1,10
14일 :: 기상 직후 복용,9
14일 :: 아침,9


In [20]:
pd.DataFrame.from_dict(frequency_by_column['other_treatment'], 
                           orient='index', 
                           columns=['frequency']).sort_values(by='frequency', ascending=False)  

,frequency
교근+측두근 보톡스,15
교근+측두근 보톡스 시술,9
악관절 고착 해소술,9
보톡스 시술(교근+측두근),5
악관절강세척술,4
보톡스 시술,2
초음파 주사 유도 (교근+측두근),2
악관절 강세척술,1
초음파 주사 유도,1
